In [1]:
from fontTools.misc.cython import returns
%load_ext autoreload
%autoreload 2

In [2]:

import os
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion

import json

api_key = os.getenv('EMBEDDING_API_KEY')
collection_name = "vector_db_pro"
db_name = "default"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")

SELF_MODEL = "qwen3.6-plus"
openai = OpenAI(
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=api_key,
)


In [3]:
class DocumentChunk(BaseModel):
    source: str = Field(description="文件名")
    document_type: str = Field(description="文件类型")
    original_text: str = Field(description="文档")


class Chunks(BaseModel):
    chunks: list[DocumentChunk]

In [4]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents

In [5]:
documents = fetch_documents()

Loaded 76 documents


In [6]:
split_system_prompt = """
You are a professional document segmentation and classification expert. Your job is to accurately classify and logically split Insurellm-related documents to ensure correct classification, reasonable chunking, and suitability for vector database storage. Duplication across segments is allowed to preserve critical information. Follow all rules strictly to avoid dangling references, unbalanced chunk sizes, or other issues that harm retrieval accuracy.

# 1. Core Classification System (Fixed, Must Be Followed Exactly)

Documents must be classified into exactly the 4 categories below. Do not add, merge, or rename categories.

| Category  | Scope of Content                                             |
| --------- | ------------------------------------------------------------ |
| company   | Overall company information: history, structure, culture, values, hiring, scale, brand, partnership ecosystem, etc. |
| contract  | Formal contracts, agreements, terms, fees, obligations, scope of service, SLA, signatures, addenda, attachments, etc. |
| employees | Employee HR records: personal info, promotion, performance, compensation, training, discipline, labor contracts, attendance, etc. |
| products  | Product details: features, pricing, roadmap, use cases, value, modules, changelogs, tutorials, environment, etc. |

# 2. Classification & Segmentation Rules

## 2.1 Classification Rules

1. **Follow assigned category**: The user will provide the document category. Use only that category; do not change or reassign.
2. **Whole-document understanding**: Segment based on full document logic, not just local phrases.
3. **Category alignment**: All chunks must align with the assigned category. Cross-category content may be segmented separately but must remain aligned with the primary category.

## 2.2 Reference Clarity Rules (Critical for Retrieval)

1. **Explicit subject required**: Every chunk must clearly state its main subject (e.g., "Insurellm" for company docs, employee name for employee docs, product name for product docs).
2. **No dangling pronouns**: Remove ambiguous pronouns such as "the company", "it", "they", "this" unless clearly defined earlier in the same chunk. Rewrite to use explicit subjects: e.g., change "the company underwent restructuring" to "Insurellm underwent restructuring".
3. **Consistent references**: Use consistent naming within each chunk.

## 2.3 Segmentation Rules

1. **Chunk size**: Each segment must be 50–500 words (flexible to 600 words for complex content). Avoid overly small chunks (single sentences) or overly large blocks.
2. **Controlled repetition**: Key information may repeat naturally across chunks to improve retrieval. Do not add meaningless repetition.
3. **Logical flow**: Preserve original document order (contract clauses, product modules, company timeline, etc.).
4. **Cross-category content**: If content spans categories, segment it separately and label primary/secondary relevance.
5. **Clean content**: Remove noise, redundant filler, and irrelevant content while preserving all critical information.

# 3. Output Format (Strict JSON, No Exceptions)

Return only valid JSON in the structure below. No extra text, explanations, or commentary.

```json
{
  "chunks": [
    {
      "text": "Segmented plain text content, self-contained, explicit subject, no dangling references, 50-500 words, may repeat key information"
    },
    {
      "text": "Another segment following the same rules"
    }
  ]
}
```

Additional rules:

- Text must be **plain text only** — no Markdown, symbols, formatting, or special characters.
- Short documents (< 50 words) may be one single chunk.
- **DO NOT TRANSLATE ANY ENGLISH CONTENT INTO CHINESE**. Preserve the original language exactly.

# 4. Execution Requirements

1. **Rule priority**: Follow all rules strictly. No deviations in categories, references, chunk size, or format.
2. **Verifiable logic**: Each chunk must represent one clear information unit.
3. **Self-contained chunks**: Each chunk must be understandable on its own without external context.
4. **Per-document output**: One JSON per document.
5. **Final check**: Verify: (1) explicit subject, (2) valid chunk length, (3) category alignment, (4) valid JSON.

# 5. Additional Notes

1. For `company` documents: replace all instances of "the company" or "it" with "Insurellm" to eliminate ambiguity.
2. For multi-paragraph documents: segment by logical phases (founding → growth → restructuring → current state).
3. **No over-segmentation**: Do not split a single logical idea into multiple chunks.
"""

In [7]:
def make_user_prompt(document):
    return f"""
doc_type:{document["type"]}
source:{document["source"]}
text:
{document["text"]}

"""

In [8]:
def make_messages(document):
    return [
        {"role": "system", "content": split_system_prompt},
        {"role": "user", "content": make_user_prompt(document)},
    ]

In [9]:
document_one = documents[2]
messages = make_messages(document_one)


In [10]:
def process_document(document):
    messages = make_messages(document)
    completion = openai.chat.completions.create(
        model=SELF_MODEL,  # 您可以按需更换为其它深度思考模型
        messages=messages,
        response_format={"type": "json_object"},
        extra_body={"enable_thinking": False},
    )
    reply = completion.choices[0].message.content
    result = json.loads(reply)
    chunks = result["chunks"]  # 获取chunks数组
    chunk_objects = []
    for chunk in chunks:
        doc_chunk = DocumentChunk(
            source=document["source"],
            document_type=document["type"],
            original_text=chunk["text"]
        )
        chunk_objects.append(doc_chunk)

    return chunk_objects

In [91]:
one_doc_chunks = process_document(document_one)
print(document_one["text"])
print("===================================")
print(one_doc_chunks)

# Insurellm Culture

## Vision Statement
To revolutionize the insurance industry through innovative technology that makes insurance accessible, transparent, and effortless for everyone.

## Mission Statement
We empower insurance providers and consumers with cutting-edge software solutions that streamline processes, enhance customer experiences, and drive meaningful connections in the insurance marketplace. By combining deep industry expertise with technological innovation, we're building the future of insurance.

## Core Values

### Innovation First
We challenge the status quo and embrace creative problem-solving. Our team is encouraged to experiment, take calculated risks, and push the boundaries of what's possible in insurance technology. We believe that breakthrough solutions come from curiosity, collaboration, and a willingness to learn from both successes and failures.

### Customer Obsession
Our clients' success is our success. We deeply understand our customers' needs and work t

In [11]:
chunks = []  # 所有切分后的结果（持久化）

In [12]:
start_idx = 0  # 起始索引：首次=0，报错后修改为报错索引继续


In [94]:
# ===================== 批量切分主程序（失败立即停止） =====================
# documents = 你的全部文档列表（自行定义）

for i, doc in enumerate(documents):
    # 跳过已处理完成的文档（断点续跑）
    if i < start_idx:
        continue

    print(f"🔄 处理文档：索引 {i}")

    try:
        # 调用你的切分函数
        one_document_chunks = process_document(doc)

        # 追加到总结果（必须用 extend）
        chunks.extend(one_document_chunks)
        print(f"✅ 文档索引 {i} 处理成功\n")

    except Exception as e:
        # 失败：打印索引 + 错误，直接停止程序
        print(f"\n❌ 【致命错误】文档索引 {i} 处理失败！")
        print(f"错误信息：{str(e)}")
        print(f"👉 请修复文档后，将 Cell1 的 start_idx = {i} 重新运行！")

        # 立即停止，不处理下一个文档
        raise

🔄 处理文档：索引 0
✅ 文档索引 0 处理成功

🔄 处理文档：索引 1
✅ 文档索引 1 处理成功

🔄 处理文档：索引 2
✅ 文档索引 2 处理成功

🔄 处理文档：索引 3
✅ 文档索引 3 处理成功

🔄 处理文档：索引 4
✅ 文档索引 4 处理成功

🔄 处理文档：索引 5
✅ 文档索引 5 处理成功

🔄 处理文档：索引 6
✅ 文档索引 6 处理成功

🔄 处理文档：索引 7
✅ 文档索引 7 处理成功

🔄 处理文档：索引 8
✅ 文档索引 8 处理成功

🔄 处理文档：索引 9
✅ 文档索引 9 处理成功

🔄 处理文档：索引 10
✅ 文档索引 10 处理成功

🔄 处理文档：索引 11
✅ 文档索引 11 处理成功

🔄 处理文档：索引 12
✅ 文档索引 12 处理成功

🔄 处理文档：索引 13
✅ 文档索引 13 处理成功

🔄 处理文档：索引 14
✅ 文档索引 14 处理成功

🔄 处理文档：索引 15
✅ 文档索引 15 处理成功

🔄 处理文档：索引 16
✅ 文档索引 16 处理成功

🔄 处理文档：索引 17
✅ 文档索引 17 处理成功

🔄 处理文档：索引 18
✅ 文档索引 18 处理成功

🔄 处理文档：索引 19
✅ 文档索引 19 处理成功

🔄 处理文档：索引 20
✅ 文档索引 20 处理成功

🔄 处理文档：索引 21
✅ 文档索引 21 处理成功

🔄 处理文档：索引 22
✅ 文档索引 22 处理成功

🔄 处理文档：索引 23
✅ 文档索引 23 处理成功

🔄 处理文档：索引 24
✅ 文档索引 24 处理成功

🔄 处理文档：索引 25
✅ 文档索引 25 处理成功

🔄 处理文档：索引 26
✅ 文档索引 26 处理成功

🔄 处理文档：索引 27
✅ 文档索引 27 处理成功

🔄 处理文档：索引 28
✅ 文档索引 28 处理成功

🔄 处理文档：索引 29
✅ 文档索引 29 处理成功

🔄 处理文档：索引 30
✅ 文档索引 30 处理成功

🔄 处理文档：索引 31
✅ 文档索引 31 处理成功

🔄 处理文档：索引 32
✅ 文档索引 32 处理成功

🔄 处理文档：索引 33
✅ 文档索引 33 处理成功

🔄 处理文档：索引 34
✅ 文档索引 34 处理成功

🔄 处理文

In [95]:
#将切分好的数据chunks写入到文件中防止数据丢失
import json
from pathlib import Path

# 保存文件路径（自动存在当前目录，名字可自定义）
SAVE_PATH = "processed_chunks_backup1.json"

# 把 DocumentChunk 对象转换成可保存的字典列表
chunks_data = [chunk.model_dump() for chunk in chunks]

# 写入本地文件
with open(SAVE_PATH, "w", encoding="utf-8") as f:
    json.dump(chunks_data, f, ensure_ascii=False, indent=2)

print(f"✅ 所有chunks已成功保存到本地文件：{SAVE_PATH}")
print(f"📊 总保存数量：{len(chunks_data)} 条")

✅ 所有chunks已成功保存到本地文件：processed_chunks_backup.json
📊 总保存数量：557 条


In [ ]:
#读取数据

import json
from pathlib import Path
from pydantic import BaseModel, Field


# 你的DocumentChunk类（必须定义才能恢复）
class DocumentChunk(BaseModel):
    source: str = Field(description="文件名")
    document_type: str = Field(description="文件类型")
    original_text: str = Field(description="文档")


# 读取文件并恢复成对象列表
SAVE_PATH = "processed_chunks_backup.json"
with open(SAVE_PATH, "r", encoding="utf-8") as f:
    chunks_data = json.load(f)

# 恢复到全局变量 chunks
chunks = [DocumentChunk(**item) for item in chunks_data]

print(f"✅ 从本地文件恢复成功！")
print(f"📊 恢复chunk总数：{len(chunks)} 条")

In [104]:
one_chunk = chunks[0]

In [105]:
import dashscope


def get_embedding(text):
    resp = dashscope.TextEmbedding.call(
        model="text-embedding-v4",
        input=[one_chunk.original_text],
        api_key=api_key,
        dimension=2048,
        text_type="document"  #query 和 document
    )

    if resp.status_code != 200:
        raise Exception(f"向量生成失败：{resp.message}")

    return resp.output["embeddings"][0]["embedding"]

In [106]:
one_chunk_embedding = get_embedding(one_chunk.original_text)

In [111]:
from pymilvus import MilvusClient

v_collection_name = "vector_db_pro_max"
client = MilvusClient(
    uri="http://192.168.175.142:19530",
    db_name="default",
)

datas = []
for chunk in chunks:
    data = {
        "text": chunk.original_text,
        "doc_type": chunk.document_type,
        "source": chunk.source,
        "vector": get_embedding(chunk.original_text)
    }
    datas.append(data)



In [113]:
client.insert(
    collection_name=v_collection_name,
    data=datas
)

{'insert_count': 557, 'ids': [465630852002348117, 465630852002348118, 465630852002348119, 465630852002348120, 465630852002348121, 465630852002348122, 465630852002348123, 465630852002348124, 465630852002348125, 465630852002348126, 465630852002348127, 465630852002348128, 465630852002348129, 465630852002348130, 465630852002348131, 465630852002348132, 465630852002348133, 465630852002348134, 465630852002348135, 465630852002348136, 465630852002348137, 465630852002348138, 465630852002348139, 465630852002348140, 465630852002348141, 465630852002348142, 465630852002348143, 465630852002348144, 465630852002348145, 465630852002348146, 465630852002348147, 465630852002348148, 465630852002348149, 465630852002348150, 465630852002348151, 465630852002348152, 465630852002348153, 465630852002348154, 465630852002348155, 465630852002348156, 465630852002348157, 465630852002348158, 465630852002348159, 465630852002348160, 465630852002348161, 465630852002348162, 465630852002348163, 465630852002348164, 4656308520

In [8]:
from week5.milvus_op import get_all
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go

all_data = get_all()
milvus_vectors = [v["vector"] for v in all_data]
np_vectors = np.array(milvus_vectors)

milvus_documents = [d["text"] for d in all_data]
milvus_doc_type = [dt["doc_type"] for dt in all_data]
milvus_colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in
                 milvus_doc_type]

# np_vectors = np.array(milvus_vectors, dtype=np.float32)

# 🔍 必须做数据校验！避免数据错误导致卡死
print(f"✅ 原始向量形状: {np_vectors.shape}")  # 必须是 (557, 2048)
print(f"✅ 向量数据类型: {np_vectors.dtype}")  # 应为 float32
print(f"✅ 是否包含NaN: {np.isnan(np_vectors).any()}")  # 必须为 False


✅ 原始向量形状: (970, 1024)
✅ 向量数据类型: float64
✅ 是否包含NaN: False


In [9]:
# ===================== 直接接在你现有代码后运行 =====================
# 1. 优化版 t-SNE（直接用2048维float32向量，无PCA，速度快）
tsne = TSNE(
    n_components=2,
    perplexity=40,      # 适配557条数据，让点更分散
    max_iter=800,      # 充分迭代，优化分布
    init="random",     # 随机初始化，避免点过度集中
    method="barnes_hut",
    angle=0.5,
    random_state=42,
    verbose=1,
    n_jobs=-1
)

# 计算降维结果
reduced_vectors = tsne.fit_transform(np_vectors)

# 🔥 核心修复：添加微小抖动，强行分离重叠的点（解决只显示几个点的问题）
np.random.seed(42)
reduced_vectors += np.random.normal(0, 0.8, reduced_vectors.shape)

# 2. 绘图（完全复用你定义的 milvus_colors / documents / doc_type）
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    # 优化点样式：大尺寸+白色描边，清晰区分重叠点
    marker=dict(size=7, color=milvus_colors, opacity=0.8, line=dict(width=1, color='white')),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(milvus_doc_type, milvus_documents)],
    hoverinfo='text'
)])

# 修复布局：删除3D专属的scene，2D图正确配置
fig.update_layout(
    title='2D Milvus Vector Visualization',
    title_x=0.5,
    xaxis_title='t-SNE 1',
    yaxis_title='t-SNE 2',
    width=1000,
    height=700,
    margin=dict(r=20, b=10, l=10, t=40),
    showlegend=False  # 你已用颜色区分类型，关闭默认图例更简洁
)

# 显示图像
fig.show()

[t-SNE] Computing 121 nearest neighbors...
[t-SNE] Indexed 970 samples in 0.002s...
[t-SNE] Computed neighbors for 970 samples in 0.064s...
[t-SNE] Computed conditional probabilities for sample 970 / 970
[t-SNE] Mean sigma: 0.307485
[t-SNE] KL divergence after 250 iterations with early exaggeration: 64.911018
[t-SNE] KL divergence after 800 iterations: 0.924500


In [10]:
# 3D t-SNE 降维（优化参数，速度快 + 点分散）
tsne = TSNE(
    n_components=3,
    perplexity=40,      # 适配557条数据，分散点
    max_iter=800,       # 充分迭代
    init="random",
    method="barnes_hut",
    random_state=42,
    verbose=1,
    n_jobs=-1
)
reduced_vectors = tsne.fit_transform(np_vectors)

# 🔥 核心：3D坐标抖动，分离重叠点（解决只显示几个点的问题）
np.random.seed(42)
reduced_vectors += np.random.normal(0, 0.6, reduced_vectors.shape)

# 3D散点图（复用你所有变量）
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    # 优化样式：白色描边，区分重叠点
    marker=dict(size=6, color=milvus_colors, opacity=0.8, line=dict(width=1, color='white')),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(milvus_doc_type, milvus_documents)],
    hoverinfo='text'
)])

# 3D布局（保留你的原版配置，无修改）
fig.update_layout(
    title='3D Milvus Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

[t-SNE] Computing 121 nearest neighbors...
[t-SNE] Indexed 970 samples in 0.002s...
[t-SNE] Computed neighbors for 970 samples in 0.050s...
[t-SNE] Computed conditional probabilities for sample 970 / 970
[t-SNE] Mean sigma: 0.307485
[t-SNE] KL divergence after 250 iterations with early exaggeration: 64.925804
[t-SNE] KL divergence after 800 iterations: 0.877561
